In [32]:
import os
from dotenv import load_dotenv
from datasets import Dataset
from pandas import DataFrame
from langfuse import get_client, Langfuse


In [34]:
import langfuse
print(langfuse.__version__)

4.14.2


In [33]:
load_dotenv()

# Verify the key is loaded (don't print the actual key in output!)
if not os.getenv("LANGFUSE_SECRET_KEY") or not os.getenv("LANGFUSE_PUBLIC_KEY"):
    raise ValueError("LANGFUSE_SECRET_KEY or LANGFUSE_PUBLIC_KEY not found. Please check your .env file.")

if not os.getenv("LANGFUSE_HOST"):
    raise ValueError("LANGFUSE_HOST not found in the .env")


In [39]:
# Initialize Langfuse client with an explicit URL
public_key = os.getenv("LANGFUSE_PUBLIC_KEY")
secret_key = os.getenv("LANGFUSE_SECRET_KEY")
# langfuse2 = Langfuse(
#   public_key=public_key,
#   secret_key=secret_key,
#   host="http://localhost:3000" # <--- Specify the URL here
# )
#
# langfuse2.auth_check()
# traces = langfuse2.get_traces(
#     limit=100
# )

AttributeError: 'Langfuse' object has no attribute 'get_traces'

In [44]:
import requests

r = requests.get(
    "http://localhost:3000/api/public/traces",
    auth=(public_key, secret_key)
)

print(r.status_code)
# print(r.json())
import json
data = r.json()
print(f"Records:  {len(data)}")
print(json.dumps(data, indent=2))

200
Records:  2
{
  "data": [
    {
      "id": "7a889173580c10ec91f69bad94fb7490",
      "projectId": "cmraii35v0006p907s7rgfk05",
      "name": "litellm_request",
      "timestamp": "2026-08-01T15:28:04.767Z",
      "environment": "production",
      "tags": [],
      "bookmarked": false,
      "release": null,
      "version": null,
      "userId": null,
      "sessionId": null,
      "public": false,
      "input": [
        {
          "type": "message",
          "role": "system",
          "content": "You are an HR Assistant.  \n\nYou are given the Context and a Question.\n\nAnswer ONLY from the supplied context.  \n\nIf the answer cannot be found in the context, say:  \"I couldn't find this information in the policies.\"  \n\nNever invent policies.\n\nQuote relevant sections whenever possible."
        },
        {
          "type": "message",
          "role": "user",
          "content": "user query: "
        },
        {
          "type": "message",
          "role": "assis

In [37]:
langfuse = get_client(public_key=public_key)

In [45]:

obs = langfuse.api.legacy.observations_v1.get_many(
    limit=20
)

print(obs)


data=[ObservationsView(id='3d931a2120d04870', trace_id='7a889173580c10ec91f69bad94fb7490', type='GENERATION', name='litellm_request', start_time=datetime.datetime(2026, 8, 1, 15, 28, 4, 767000, tzinfo=TzInfo(0)), end_time=datetime.datetime(2026, 8, 1, 15, 28, 5, 388000, tzinfo=TzInfo(0)), completion_start_time=None, model='llama-3.3-70b-versatile', model_parameters={'temperature': 0.3, 'stream': 'false', 'tools': '[]'}, input=[{'type': 'message', 'role': 'system', 'content': 'You are an HR Assistant.  \n\nYou are given the Context and a Question.\n\nAnswer ONLY from the supplied context.  \n\nIf the answer cannot be found in the context, say:  "I couldn\'t find this information in the policies."  \n\nNever invent policies.\n\nQuote relevant sections whenever possible.'}, {'type': 'message', 'role': 'user', 'content': 'user query: '}, {'type': 'message', 'role': 'assistant', 'content': "You haven't asked a question yet. Please go ahead and ask, and I'll do my best to help!"}, {'type': '

In [82]:
o = obs.data[0]
print(type(o))
print(type(o.metadata))
print(o.name)
print(o.trace_id)
md = o.metadata['attributes']['metadata']
print(type(md))

meta = json.loads(md)
print(type(meta))

print(meta['requester_metadata'])

<class 'langfuse.api.commons.types.observations_view.ObservationsView'>
<class 'dict'>
litellm_request
7a889173580c10ec91f69bad94fb7490
<class 'str'>
<class 'dict'>
{'request_id': '1866', 'session_id': 'ABC123', 'request_time': '2026-08-01T20:57:58.824+05:30', 'route': 'hr', 'langfuse.trace.metadata.route': 'hr', 'langfuse.trace.metadata.request_id': '1866', 'langfuse.session.id': '1866', 'langfuse': {'trace': {'name': 'YOYO'}}}


In [48]:

print(dir(langfuse.api.legacy))


print(dir(langfuse.api))
print()
print(dir(langfuse.api.legacy))


['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', '_client_wrapper', '_metrics_v1', '_observations_v1', '_raw_client', '_score_v1', 'metrics_v1', 'observations_v1', 'score_v1', 'with_raw_response']
['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', '_annotation_queues', '_blob_storage_integ

In [83]:
# Method 1: Score via low-level method
langfuse.create_score(
    name="faithfullness",
    value=0.8,
    trace_id="7a889173580c10ec91f69bad94fb7490",
    #observation_id="observation_id_here", # optional
    data_type="NUMERIC", # optional, inferred if not provided
    comment="Grounded", # optional
)

In [11]:
# 1. Fetch a batch of recent traces
# (Strongly recommended: Use tags, session_id, or time limits to avoid fetching your entire database)
traces_response = langfuse.fetch_traces(
    limit=100,
    tags=["n8n_agent"] # Narrow down the payload
)

target_request_id = "your_target_id_here"
found_trace_id = None

# 2. Iterate and filter client-side
for trace in traces_response.data:
    # Safely traverse the nested metadata dictionary
    metadata = trace.metadata or {}
    req_meta = metadata.get("requester_metadata", {})

    # Check your nested field
    if req_meta.get("Itemsrequest_id") == target_request_id:
        found_trace_id = trace.id
        break

if found_trace_id:
    print(f"Found corresponding Trace ID: {found_trace_id}")
    # You can now fetch the full trace detail if needed: langfuse.fetch_trace(found_trace_id)
else:
    print("Trace not found.")

Context error: No active span in current context. Operations that depend on an active span will be skipped. Ensure spans are created with start_as_current_observation() or that you're operating within an active span context.
